# 2. The Sequential Chain

A **sequential chain** is a simple chain grown up: **multiple stages in a row**, where each stage's
output becomes the next stage's input. Summarize → translate → extract keywords. It's the same `|`
idea, just longer.

---

## 1. Simple Definition

> **Kid version:** It's a relay race 🏃‍♂️➡️🏃‍♀️. Runner 1 finishes and hands the baton to Runner 2,
> who runs and hands it to Runner 3. Each runner starts where the last one left off. In a sequential
> chain, the "baton" is the data.

**Professional definition:** A sequential chain is a `RunnableSequence` of two or more sub-chains/steps
executed **in order**, where the output of step *N* is passed as the input to step *N+1*. It models
multi-stage workflows that must happen one after another because each stage **depends on** the previous
one's result.

```python
# Stage 1: summarize → Stage 2: translate the summary to French
chain = summarize_chain | translate_chain
chain.invoke({"text": long_article})     # → French summary
```

---

## 2. Why Does It Exist?

**The problem:** Many tasks are **multi-step and dependent**: you can't translate a summary before you
*have* the summary. You need stage 2 to receive stage 1's output.

### Before (manual passing between stages)

```python
summary = summarize_chain.invoke({"text": article})      # step 1
translated = translate_chain.invoke({"summary": summary}) # step 2 (feed step 1's output by hand)
keywords = keyword_chain.invoke({"text": translated})     # step 3
# You're manually shuttling outputs into the next call. Error-prone, verbose.
```

### After (one sequential chain)

```python
chain = summarize_chain | translate_chain | keyword_chain
chain.invoke({"text": article})     # all three stages, output auto-piped
```

Sequential chaining automates the hand-off and gives the whole multi-stage pipeline streaming/batching/
async.

**Where you'll use it:** summarize-then-analyze, generate-then-critique-then-refine, extract-then-
validate, translate pipelines, any multi-step reasoning workflow.

---

## 3. Real-Life Analogy

**An assembly line with dependent stations** 🏭. The painter can't paint a part until the molder has
shaped it; the packer can't pack it until it's painted. Each station **must** wait for the previous
one — that ordered dependency is exactly a sequential chain.

Other analogies: a **recipe** (chop → cook → season → plate, in order), an **essay process** (outline →
draft → edit).

---

## 4. Where It Fits in LangChain Architecture

```
   RunnableSequence  (the sequential engine — same one `|` builds)
        │
        ▼
   step1 ─► step2 ─► step3 ─► ...      each step is itself a Runnable
                                       (often a whole prompt|model|parser sub-chain)
```

A simple chain is already a `RunnableSequence` of 3 steps. A "sequential chain" is the same
class with **sub-chains** as steps — the difference is one of scale/intent, not a new mechanism.

---

## 5. Internal Working — the baton passing

```
  chain = summarize | translate | keywords
  chain.invoke({"text": article})

  ① {"text": article}
        │  summarize sub-chain (prompt|model|parser)
        ▼
  ② "A short summary of the article."         ← output of stage 1
        │  becomes the INPUT to stage 2
        ▼
  ③ translate sub-chain runs on that summary
        ▼
  ④ "Un court résumé..."                       ← output of stage 2 → input to stage 3
        │
        ▼
  ⑤ keywords sub-chain → ["résumé", "article", ...]
```

**The critical detail: matching shapes.** Stage 2 must accept whatever stage 1 emits. If stage 1
outputs a **string** but stage 2's prompt expects a dict like `{"summary": ...}`, you insert a tiny
glue step (a `RunnableLambda` or a dict) to reshape it.

```python
# Reshape a plain string into the dict the next prompt expects:
chain = summarize_chain | (lambda s: {"summary": s}) | translate_chain
```

---

## 6. Building a sequential chain — worked example

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Stage 1: make a summary from {text}
summarize = (
    ChatPromptTemplate.from_template("Summarize this in 2 sentences:\n\n{text}")
    | model | StrOutputParser()
)

# Stage 2: translate whatever comes in ({summary}) to French
translate = (
    ChatPromptTemplate.from_template("Translate to French:\n\n{summary}")
    | model | StrOutputParser()
)

# Glue: stage 1 outputs a str; stage 2's prompt wants {"summary": ...}
sequential = summarize | (lambda s: {"summary": s}) | translate

print(sequential.invoke({"text": "LangChain lets you build LLM apps by composing..."}))
# → French translation of the 2-sentence summary
```

---

## 7. Key parts / patterns

### `Ordered steps` (the sequence)

**Definition:** Two or more Runnables composed with `|`, executed front-to-back.

**Why it exists:** To express *dependent* multi-stage work.

**When developers use it:** Whenever step B needs step A's result.

```python
chain = step_a | step_b | step_c
```

---

### `Glue between stages` (shape-matching)

**Definition:** A small `RunnableLambda`/dict that reshapes one stage's output into the next stage's
expected input.

**Why it exists:** Stages often speak slightly different "shapes" (str vs dict); glue bridges them.

**When developers use it:** Whenever output type ≠ next input type.

```python
chain = step_a | (lambda x: {"key": x}) | step_b
```

---

### Keeping earlier values with RunnablePassthrough.assign (preview)

**Definition:** Carry forward earlier data while adding new fields, so later stages can see *both*.

**Why it exists:** Sometimes stage 3 needs the *original* input **and** stage 2's output.

**When developers use it:** Multi-stage chains where later steps reference earlier results.

```python
from langchain_core.runnables import RunnablePassthrough
chain = RunnablePassthrough.assign(summary=summarize) | translate_using_summary_and_original
```

---

## 8. The legacy classes (recognize, don't write)

Older LangChain had dedicated sequential classes. They **still appear in tutorials** but are superseded
by LCEL:

| Legacy class | What it did | Modern replacement |
|--------------|-------------|--------------------|
| `SimpleSequentialChain` | Single string passed from one step to the next | `chainA | chainB` |
| `SequentialChain` | Multiple named inputs/outputs across steps | `|` + `RunnablePassthrough.assign` |

```python
# Legacy (deprecated), shown only for recognition:
# from langchain.chains import SimpleSequentialChain
# overall = SimpleSequentialChain(chains=[summarize, translate])

# Modern:
overall = summarize | (lambda s: {"summary": s}) | translate
```

> Rule of thumb: **write LCEL** (`|`). Reach for `RunnablePassthrough.assign` when you need
> the multi-input behavior the old `SequentialChain` provided.

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
# step1: create the prompt templates
detailed_report_prompt = PromptTemplate(template='Generate a detailed report on {topic}',
                                        input_variables=['topic'])
pointer_summary_prompt = PromptTemplate(template='Generate a 5 pointer summary from the following text \n {text}',
                                        input_variables=['text'])

In [4]:
# step2: Initialize the language model
llm = ChatOllama(model="qwen3:8b")

In [5]:
# step3: Initialize output parser
output_parser = StrOutputParser()

In [6]:
# step4: Create the chain
chain = detailed_report_prompt | llm | output_parser | pointer_summary_prompt | llm | output_parser

In [7]:
# invoke the chain with a topic
result = chain.invoke({'topic': 'Unemployment in India'})

In [8]:
print(result)

**5-Pointer Summary of the Report on Unemployment in India**  

1. **Types and Statistics**: India faces structural, cyclical, frictional, seasonal, and underemployment, with youth unemployment at 7.5% (15–24 age group) and graduates at 18.7%. Regional disparities exist, and 45% of workers are in the informal sector, highlighting skill gaps and rural-urban divides.  

2. **Root Causes**: Structural challenges (skill mismatch, agricultural stagnation), slow industrial growth, weak labor policies, demographic pressure from a young population, and pandemic-induced job losses exacerbate unemployment, compounded by inadequate vocational training and data inaccuracies.  

3. **Economic and Social Impacts**: Unemployment stifles GDP growth, deepens poverty, and widens inequality, while driving rural-to-urban migration, youth disengagement, and gender disparities. Informal workers face stagnant wages and lack social security, worsening social instability.  

4. **Government Initiatives and Cha

In [9]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       